In [ ]:
import archetypes
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
import diskcache
from itertools import chain
from collections import Counter

import igraph as ig
import matplotlib.pyplot as plt
import seaborn.objects as so
import seaborn as sns

from utils.pandas_setup import pandas_setup
pandas_setup()
# from AA_Abstract import AA_Abstract
# from AA_Original import AA_Original

import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
pyalex.config.email = "Lawrence.Cram@anu.edu.au"
pyalex.config.max_retries = 0
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]

MY_DATA_PATH = Path('/home/lc/m/working')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
MY_CACHE_FILE = Path('/home/lc/m/.cache/econommicsbusiness/cache.db')
DATAFILES_PATH = Path('../DATAFILES')

In [ ]:
class SetUp:

    def __init__(self):
        self._setup_db()
        self._setup_cache()
        return
    
    def _setup_db(self):
        self.db = duckdb.connect()
        self.db.sql(f"ATTACH IF NOT EXISTS '{MY_DATABASE_FILE}' AS project")
        self.db.sql(""" SET memory_limit = '56GB';
                        SET threads = 6;
                        SET preserve_insertion_order = false;
                        SET order_by_non_integer_literal=true;
                        SET enable_progress_bar = true;
                        SET temp_directory = '/home/lc/m/.tmp';
                    """)
        self.db.sql("SHOW ALL TABLES").show()
        return
    
    def _setup_cache(self):
        self.cache = diskcache.Cache(MY_CACHE_FILE, size_limit=16_000_000_000)
        print(f'{self.cache.check() = }')
        print(f'{self.cache.volume() = }')
        return
    
    def name_of_global_obj(self, obj=None):
        for objname, oid in globals().items():
            if oid is obj:
                return objname

In [ ]:
import archetypes.datasets


class ArchetypeAnalysis(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def extract_table(self):
        self.data = self.db.sql("SELECT * FROM project.references_work_summary").df()
        print(f'{self.data.shape = }\n{self.data.head()}\n{self.data.tail()}')
        return
    
    def condition_data(self):

        self.columns = ['work_id', 'sample', 'references_per_page', 'copied_percent', 'self_reference_rate', 
                        'most_referenced_author_frequency', 'pagerank_avg', 'influence_avg']
        # self.columns = ['works_count_endogenous', 'h_index', 'reputation_sources', 
        #                 'citedness', 'citations_endogenous', 'hca_endogenous', 'hca_total', 'c/h/h',
        #                 'gini_citer', 'gini_cited', 'gini_coauthors', 'group', 'author_id', 'author_name']
        # mask = [g in ['C', 'T'] or i < 2048 for i, g in zip(self.data.index, self.data['group'])]
        self.df = self.data.loc[:, self.columns].fillna(0)
        mask = [s < 15 and r < 50 and i < 5 for s, r, i in zip(self.df.references_per_page, self.df.copied_percent, self.df.influence_avg)]
        self.df = self.df.loc[mask]
        print(f'unfiltered DataFrame {self.df.shape = }\n{self.df.head()}')
        return

    def run_archetype(self):

        X = self.df.drop(columns=['work_id', 'sample']).to_numpy().astype(np.float64)
        print(f'INPUT DataFrame {X.shape = }')
        n_archetypes = 4
        self.model =  archetypes.AA(n_archetypes=n_archetypes, random_state=99)
        self.X_trans = self.model.fit(X).transform(X)
        return
    
    def report_archtypes(self):
        print(f'{self.model.n_archetypes = }')
        cols = self.columns
        self.arch = pd.DataFrame(self.model.archetypes_, columns=cols[2:], index=[f'A_{n}' for n in range(self.model.archetypes_.shape[0])])
        print(f'{self.arch.shape = }\n{self.arch.head(32)}')
        print(f'{self.X_trans.shape = }')
        self.trans = pd.DataFrame(self.X_trans, index=self.df.work_id, columns=self.arch.index)
        self.trans['work_id'] = self.trans.index
        self.trans['group'] = self.df['sample'].tolist()
        self.trans = self.trans.reset_index(drop=True)
        for archetype in self.arch.index:
            print(f'Sorted by archetype {archetype = }') 
            print(f'{self.trans.sort_values(archetype, ascending=False).shape = }\n{self.trans.sort_values(archetype, ascending=False).head(8)}\n{self.trans.sort_values(archetype, ascending=False).tail(8)}')
            self.df[archetype] = self.trans[archetype]
        return

    
    def simplex_plot(self):
        # mask = [x in ['C', 'T', 'X', 'CT'] for x in self.df['sample']]

        labels = [g if g in ["C", "T", "X", "CT"] else '' for g in self.df['sample']]
        mask = [True for x in self.data['sample']]
        X_trans_filter = self.X_trans
        point_colour=['blue' if g == 'C' else 'red' if g == 'T' else 'green' for g in self.df['sample']]
        point_size=[10 for x in self.df['sample']] #[40 if g == 'C' else 40 if g == 'T' else 1 for g in self.df['sample']]
        fig, ax = plt.subplots(1, 2, figsize=[20, 10])
        archetypes.visualization.simplex(X_trans_filter, show_direction=True, labels=labels, c=point_colour, s=point_size, ax=ax[0])
        # sns.heatmap((self.arch-self.arch.mean())/self.arch.std(), ax=ax[1])
        # archetypes.visualization.heatmap(self.X_trans, ax=ax[1], labels=[arch.index, arch.columns])
        plt.savefig('../PLOTS/aa_4.png')
        plt.show()
        return
    
    def parallel_plot(self):
        # Calculate the average values for each category
        cols = self.columns
        df = self.df[cols[2:]].sample(n=2000, random_state=42) #.set_index(['author_id', 'group'])
        print(df.head())
        df_zscore = (df - df.mean())/df.std()
        df_long = df_zscore #.reset_index()  #.melt(id_vars=['author_id', 'group'], var_name='measure', value_name='value')
        print(df_long.head())

        # Create parallel plot
        plt.figure(figsize=(8, 6))
        g = sns.lineplot(data=df_long.T,
                            dashes=False,
                            markers=True,
                            markersize=8,
                            legend=False,)

        # Add title
        plt.title('Parallel Plot')

        # Remove y-axis ticks and tick labels
        plt.yticks([])
        plt.xticks(rotation='vertical')


        # Display the chart
        plt.savefig("../PLOTS/pp.png")
        plt.show()

In [ ]:
def main():

    aa = ArchetypeAnalysis()
    aa.extract_table()
    aa.condition_data()
    aa.run_archetype()
    aa.report_archtypes()
    # aa.simplex_plot()
    aa.parallel_plot()
    
    return

In [ ]:
if __name__ == "__main__":
    main()
    print("DONE!")